## Data preprocessing

Merges RAF-DB train/test splits, renames classes, re-splits into train/val/test.

In [1]:
import os
import random
import shutil
from pathlib import Path
from tqdm import tqdm

In [2]:
SEED = 64
random.seed(SEED)

_cwd = Path(os.getcwd())
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
print("Project root:", PROJECT_ROOT)

Project root: /home/sera/Projects/image_emotion_detection


In [3]:
rafdb_source = PROJECT_ROOT / "data" / "RAF-DB" / "DATASET"
rafdb_target = PROJECT_ROOT / "data" / "dataset_split_rafdb"

RAF_VAL_FRACTION = 0.15

RAF_LABEL_MAP = {
    "1": "surprised",
    "2": "fearful",
    "3": "disgusted",
    "4": "happy",
    "5": "sad",
    "6": "angry",
    "7": "neutral",
}

assert (rafdb_source / "train").exists(), f"Expected {rafdb_source}/train"
assert (rafdb_source / "test").exists(),  f"Expected {rafdb_source}/test"

rafdb_target.mkdir(parents=True, exist_ok=True)
print("Source:", rafdb_source.resolve())
print("Target:", rafdb_target.resolve())

Source: /home/sera/Projects/image_emotion_detection/data/RAF-DB/DATASET
Target: /home/sera/Projects/image_emotion_detection/data/dataset_split_rafdb


In [4]:
rafdb_train_images = {}
rafdb_test_images = {}

for label_dir in sorted((rafdb_source / "train").iterdir()):
    if not label_dir.is_dir():
        continue
    class_name = RAF_LABEL_MAP[label_dir.name]
    rafdb_train_images[class_name] = [
        p for p in label_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

for label_dir in sorted((rafdb_source / "test").iterdir()):
    if not label_dir.is_dir():
        continue
    class_name = RAF_LABEL_MAP[label_dir.name]
    rafdb_test_images[class_name] = [
        p for p in label_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

total = 0
print("train (official):")
for cls, imgs in sorted(rafdb_train_images.items()):
    print(f"  {cls:>12}: {len(imgs)}")
    total += len(imgs)
print("test (official):")
for cls, imgs in sorted(rafdb_test_images.items()):
    print(f"  {cls:>12}: {len(imgs)}")
    total += len(imgs)
print("Total:", total)
assert total > 0, "No images found."

train (official):
         angry: 705
     disgusted: 717
       fearful: 281
         happy: 4772
       neutral: 2524
           sad: 1982
     surprised: 1290
test (official):
         angry: 162
     disgusted: 160
       fearful: 74
         happy: 1185
       neutral: 680
           sad: 478
     surprised: 329
Total: 15339


In [5]:
rafdb_split_paths = {"train": {}, "val": {}, "test": rafdb_test_images}

for class_name, images in rafdb_train_images.items():
    images = list(images)
    random.shuffle(images)
    n_val = int(len(images) * RAF_VAL_FRACTION)
    rafdb_split_paths["val"][class_name]   = images[:n_val]
    rafdb_split_paths["train"][class_name] = images[n_val:]

for split_name in ["train", "val", "test"]:
    count = sum(len(rafdb_split_paths[split_name][c]) for c in rafdb_split_paths[split_name])
    print(f"{split_name:>5}: {count}")

train: 10434
  val: 1837
 test: 3068


In [6]:
for split_name in ["train", "val", "test"]:
    for class_name, img_list in rafdb_split_paths[split_name].items():
        out_dir = rafdb_target / split_name / class_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for img_path in tqdm(img_list, desc=f"{split_name}/{class_name}", leave=False):
            out_path = out_dir / img_path.name
            if not out_path.exists():
                shutil.copy(img_path, out_path)

print("Done.")


train/surprised:   0%|          | 0/1097 [00:00<?, ?it/s]


train/fearful:   0%|          | 0/239 [00:00<?, ?it/s]


train/disgusted:   0%|          | 0/610 [00:00<?, ?it/s]


train/happy:   0%|          | 0/4057 [00:00<?, ?it/s]


train/happy:  31%|███▏      | 1268/4057 [00:00<00:00, 12671.92it/s]

...


test/happy:   0%|          | 0/1185 [00:00<?, ?it/s]


test/sad:   0%|          | 0/478 [00:00<?, ?it/s]


test/angry:   0%|          | 0/162 [00:00<?, ?it/s]


test/neutral:   0%|          | 0/680 [00:00<?, ?it/s]

Done.


In [7]:
for split_name in ["train", "val", "test"]:
    count = sum(
        len(list((rafdb_target / split_name / c).glob("*.*")))
        for c in rafdb_split_paths[split_name]
    )
    print(f"{split_name:>5}: {count} files")

train: 10434 files
  val: 1837 files
 test: 3068 files
